Query definition of selected wikipedia data from OpenAI ChatGPT

In [1]:
# Create client
from openai import OpenAI
client = OpenAI()

In [30]:
# Load data
import pickle
with open("selected_data.bin", "rb") as f:
    data = pickle.load(f)

In [31]:
# Show data
data

[{'title': 'Creative computing',
  'text': 'Creative computing covers the interdisciplinary area at the cross-over of the creative arts and computing. Issues of creativity include knowledge discovery, for example.'},
 {'title': 'Animal-computer interaction',
  'text': 'Animal-computer interaction (ACI) is a field of research for the design and use of technology with, for and by animals. It emerged from, and is heavily influenced by, the discipline of human-computer interaction (HCI).'},
 {'title': 'Frame problem',
  'text': 'In artificial intelligence, the frame problem describes an issue with using first-order logic (FOL) to express facts about a robot in the world. Representing the state of a robot with traditional FOL requires the use of many axioms that simply imply that things in the environment do not change arbitrarily. For example, Hayes describes a "block world" with rules about stacking blocks together. In a FOL system, additional axioms are required to make inferences about 

In [32]:
len(data)

1248

In [2]:
# Test query
response = client.responses.create(
    model="gpt-5-nano",
    input = [
        {
            "role": "system",
            "content": "Give a short description of the provided term (1-2 paragraphs). Don't ask the user anything."
        },
        {
            "role": "user",
            "content": "Creative computing"
        }
    ]
)

print(response.output_text)

Creative computing is an interdisciplinary field that treats computing as a material for creative expression, rather than just a tool for number crunching. It blends computer science with art, design, music, and performance, focusing on using software, hardware, and algorithms to produce expressive, interactive works. It emphasizes experimentation, prototyping, and collaboration between technologists and artists to explore new forms of inquiry and communication.

Practically, it encompasses generative art, interactive installations, digital storytelling, game design, data visualization as aesthetic practice, and physical computing with sensors and microcontrollers. Common tools include Processing and p5.js, Python, Arduino, and Raspberry Pi, while approaches range from real-time interaction and procedural generation to AI-assisted creativity. The goal is to democratize creation, expand how we express ideas with technology, and cultivate computational thinking through hands-on, imaginat

In [ ]:
# Prepare batch file
identifier = 0
requests = []
for entry in data:
    identifier += 1
    request = {
        "custom_id": f"request-{identifier}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-5.1",
            "messages": [
                {
                    "role": "system",
                    "content": "Give a short description of the provided term (1-2 paragraphs). Don't ask the user anything.",
                }, {
                    "role": "user",
                    "content": entry["title"]
                },
            ]
        }
    }
    requests.append(request)

In [50]:
# Store batch file
import json

with open("batchinput.jsonl", "wb") as f:
    for request in requests:
        line = json.dumps(request, ensure_ascii=False) + "\n"
        f.write(line.encode("utf-8"))

In [51]:
# Upload batch file
batch_input_file = client.files.create(
    file = open("batchinput.jsonl", "rb"),
    purpose="batch"
)

In [52]:
print(batch_input_file)

FileObject(id='file-2tQB6YFymvgazgH8LEjy4K', bytes=14986, created_at=1766011143, filename='batchinput.jsonl', object='file', purpose='batch', status='processed', expires_at=1768603143, status_details=None)


In [53]:
# Create batch
batch_input_file_id = batch_input_file.id
batch = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "batch mode test"
    }
)

In [56]:
# Check batch status
status = client.batches.retrieve(batch.id) # Id is from output above
print(status)
print(status.status)

Batch(id='batch_6943310d9bcc8190872324baca230c8a', completion_window='24h', created_at=1766011149, endpoint='/v1/chat/completions', input_file_id='file-2tQB6YFymvgazgH8LEjy4K', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1766011193, error_file_id=None, errors=None, expired_at=None, expires_at=1766097549, failed_at=None, finalizing_at=1766011186, in_progress_at=1766011151, metadata={'description': 'batch mode test'}, model='gpt-5.1-2025-11-13', output_file_id='file-4ThhdpGhHybx2MugQZ91hb', request_counts=BatchRequestCounts(completed=50, failed=0, total=50), usage=BatchUsage(input_tokens=1682, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=7926, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=9608))
completed


In [ ]:
# Get error if there is an error
error_file_id = status.error_file_id
errors = client.files.content(error_file_id)
error_bytes = errors.read()
print(error_bytes.decode("utf-8"))

from https://platform.openai.com/docs/guides/batch :

The status of a given Batch object can be any of the following:

| Status     	| Description |
|---------------|-------------|
| validating	| the input file is being validated before the batch can begin |
| failed	    | the input file has failed the validation process |
| in_progress	| the input file was successfully validated and the batch is currently being run |
| finalizing	| the batch has completed and the results are being prepared |
| completed	    | the batch has been completed and the results are ready |
| expired	    | the batch was not able to be completed within the 24-hour time window |
| cancelling	| the batch is being cancelled (may take up to 10 minutes) |
| cancelled	    | the batch was cancelled |

In [60]:
# Retrieve results
file_response = client.files.content(status.output_file_id)

In [62]:
file_response.text

'{"id": "batch_req_69433133e1808190a64988814f035162", "custom_id": "request-1", "response": {"status_code": 200, "request_id": "bff648715d4c777c5a2384f81b7b7ceb", "body": {"id": "chatcmpl-CnuSvrwen6Ymw6wNbEHPTMROz8jG1", "object": "chat.completion", "created": 1766011161, "model": "gpt-5.1-2025-11-13", "choices": [{"index": 0, "message": {"role": "assistant", "content": "Creative computing is the use of computer technology as a medium for artistic expression, design, and imaginative problem-solving. It blends programming, digital tools, and interactive media with creativity to produce things like generative art, interactive installations, digital storytelling, games, and creative data visualizations.\\n\\nRather than focusing solely on efficiency or traditional software applications, creative computing emphasizes experimentation, play, and new forms of human\\u2013computer interaction. It often draws on fields such as art, design, music, and media studies, and encourages people\\u2014of

In [65]:
# Store result
with open("batchoutput.jsonl", "w") as f:
    f.write(file_response.text)